---
# <div align="center"><font color='red'>  </font></div>
# <div align="center"><font color='red'> COSC 2779/2972 | Deep Learning  </font></div>
## <div align="center"> <font color='red'> Week 6 Lectorial Example: **Practical methodology, Hyperparameter tuning & Transfer learning**</font></div>
---

# The methodology behind this notebook

We are going to follow the iterative workflow from Andrew Ng's *Machine Learning Yearning*. Every design decision in the rest of this notebook maps to one of these four steps:

1. **Establish a goal** — pick a single performance metric and a target value. Without a number to aim at, "better" is undefined.
2. **Establish a baseline** — not a trivial model, but a model that has a *reasonable chance of success* on the task. The baseline anchors every future comparison.
3. **Instrument** — track training/validation loss and the metric of interest as curves, not just a final number. Learning curves are how we diagnose what to do next.
4. **Iterate using the signals** — read the curves. Overfitting? Add regularisation / more data. Underfitting? Bigger model / longer training / better features. Change one thing at a time so you know what caused the improvement.

Keep coming back to this list as you read the notebook.

# Task

Assume that you are working for a startup that is designing pet feeders.

Your task is to design a "Pet feeding machine" that dispenses the appropriate food type based on whether a cat or a dog is nearby.

There are products that do this using things like ID tags. However, you want to design a more convenient product that uses computer vision to identify if a cat or a dog is nearby.

This notebook only concentrates on the CV side.

## Identify some ground rules (requirements)

- Let's assume you are going to use still photographs coming from an RGB camera.

- You have some **computing power** onboard the device and want to use that to do the processing. No connection to internet. The computing power is equal to a modern phone.

- What sort of **performance** are we after?

- **Data**: how to obtain data, how much and what type?


# Approach 1

## Data

Look for some data that can support your task.

Closest PyTorch-friendly dataset is the **Oxford-IIIT Pet dataset**, which torchvision provides out of the box with a `binary-category` target (0 = cat, 1 = dog). It's ~7.4k images.

The first time you run this cell it will download ~800 MB into `./data`. Subsequent runs are cached.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset

from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
torch.manual_seed(42)

class_label = ['cat', 'dog']


In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32

# For the moment we just resize + convert to tensor; augmentation & normalisation added later.
raw_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# torchvision's OxfordIIITPet has native binary cat/dog labels.
full_trainval = OxfordIIITPet(
    root='./data', split='trainval', target_types='binary-category',
    download=True, transform=raw_transform,
)
test_dataset = OxfordIIITPet(
    root='./data', split='test', target_types='binary-category',
    download=True, transform=raw_transform,
)

# Mirror the original TF split proportions (40% train / 10% val / 10% test of the whole).
# torchvision already gives us a trainval and test set, so we further split trainval into train/val.
g = torch.Generator().manual_seed(42)
indices = torch.randperm(len(full_trainval), generator=g).tolist()
n_val = int(0.2 * len(indices))
train_ds = Subset(full_trainval, indices[n_val:])
val_ds   = Subset(full_trainval, indices[:n_val])
test_ds  = test_dataset

print(f'Number of training samples:   {len(train_ds)}')
print(f'Number of validation samples: {len(val_ds)}')
print(f'Number of test samples:       {len(test_ds)}')


Observation:
- Limited training dataset.

Let's visualise some images.

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(9):
    image, label = train_ds[i]                  # image is (3, 160, 160) float in [0,1]
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image.permute(1, 2, 0).numpy())
    plt.title(class_label[int(label)])
    plt.axis('off')
plt.show()


Observations:

- Some images are from pets in indoor settings — the images we want for our product. There are a lot of images from irrelevant environments.

- Most animals are in poses that are not relevant.

- Minor: original images have different shapes (we've already resized them to 160×160).

Let's observe class distribution.

In [ ]:
train_labels = [int(full_trainval._labels[i]) for i in train_ds.indices]  # binary-category label
plt.hist(train_labels, bins=[-0.5, 0.5, 1.5])
plt.xticks([0, 1], class_label)
plt.ylabel('count')
plt.show()


Observation:
- Classes are equally represented.

## Step 1 — Determine the goal

For this task we need to achieve around **90% accuracy**, with **similar performance in both classes** (no severe class imbalance in errors).

- Use **cross-entropy** as loss and **accuracy** as main metric.
- Observe other metrics like **recall**, **precision**, and **AUC** to make sure the performance is balanced.

Writing this down explicitly *before* training anything is what stops "hyperparameter tuning" from degenerating into random search.

## Step 2 — Establish a default baseline model

Since we are going to run the final model on a low-power device we need to consider the model's footprint.

Let's use **MobileNetV2** — a network architecture developed by Google that is computationally efficient.

[MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications](https://arxiv.org/abs/1704.04861)

For **Approach 1** we deliberately instantiate MobileNetV2 with **random weights** — no ImageNet pretraining. We want to see how far we get training from scratch on our ~5.9k images. Spoiler: this is not exactly the "reasonable-effort baseline" from Ng's book. But, It should give us a *signal* about whether the problem is even tractable and what direction to move in.

In [ ]:
def build_scratch_model(num_classes=2):
    m = mobilenet_v2(weights=None)                # random init
    # Original classifier is Dropout(0.2) -> Linear(1280, 1000); replace with our head.
    m.classifier = nn.Sequential(
        nn.Linear(1280, num_classes),
    )
    return m

model = build_scratch_model()
n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params:,}')


### Preprocessing + augmentation

MobileNetV2's ImageNet weights expect input normalised with the standard ImageNet mean/std. For consistency we use the same normalisation whether we train from scratch or transfer.

We also add light data augmentation on the *training* pipeline only (horizontal flip + small rotation) — the same as the original.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Rebuild datasets so train uses augmentation and val/test don't.
train_full_aug = OxfordIIITPet(root='./data', split='trainval', target_types='binary-category',
                               download=False, transform=train_transform)
val_full_eval  = OxfordIIITPet(root='./data', split='trainval', target_types='binary-category',
                               download=False, transform=eval_transform)
test_ds_eval   = OxfordIIITPet(root='./data', split='test',     target_types='binary-category',
                               download=False, transform=eval_transform)

train_ds = Subset(train_full_aug, indices[n_val:])
val_ds   = Subset(val_full_eval,  indices[:n_val])
test_ds  = test_ds_eval

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)


## Step 3 — Set up instrumentation

We track train + validation loss and all four metrics (accuracy / precision / recall / AUC) per epoch. `train_and_track` returns a Keras-style history dict so we can layer multiple experiments onto the same plot with `plotter`.

In [ ]:
from itertools import cycle

def plotter(history_hold, metric='loss', ylim=None):
    cycol = cycle('bgrcmk')
    for name, hist in history_hold.items():
        y_train = hist[metric]
        y_val   = hist['val_' + metric]
        x = np.arange(len(y_val))
        c = next(cycol)
        plt.plot(x, y_train, c + '-',  label=name + '_train')
        plt.plot(x, y_val,   c + '--', label=name + '_val')
    plt.legend()
    plt.xlim([1, max(plt.xlim())])
    if ylim is not None:
        plt.ylim(ylim)
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.grid(True)

m_histories = {}


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, ys, preds, probs_pos = 0.0, [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(xb)
            p_pos = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            ys.extend(yb.cpu().numpy())
            preds.extend(logits.argmax(1).detach().cpu().numpy())
            probs_pos.extend(p_pos)
    n = len(ys)
    metrics = {
        'loss':      total_loss / n,
        'accuracy':  accuracy_score(ys, preds),
        'precision': precision_score(ys, preds, zero_division=0),
        'recall':    recall_score(ys, preds, zero_division=0),
        'auc':       roc_auc_score(ys, probs_pos) if len(set(ys)) > 1 else float('nan'),
    }
    return metrics


def train_and_track(model, train_loader, val_loader, epochs=10, lr=1e-3,
                    optimizer=None, verbose=True):
    model = model.to(device)
    if optimizer is None:
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()

    hist = {k: [] for k in ['loss', 'accuracy', 'precision', 'recall', 'auc',
                             'val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'val_auc']}
    for ep in range(epochs):
        tr = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        vl = run_epoch(model, val_loader,   criterion, optimizer=None)
        for k, v in tr.items():        hist[k].append(v)
        for k, v in vl.items():        hist['val_' + k].append(v)
        if verbose:
            print(f'ep {ep+1:2d}/{epochs}  '
                  f'loss {tr["loss"]:.3f}  acc {tr["accuracy"]:.3f}  |  '
                  f'val_loss {vl["loss"]:.3f}  val_acc {vl["accuracy"]:.3f}  '
                  f'val_prec {vl["precision"]:.3f}  val_rec {vl["recall"]:.3f}  val_auc {vl["auc"]:.3f}')
    return hist


## Train model

In [ ]:
epochs = 10
m_histories['baseModel'] = train_and_track(model, train_loader, val_loader, epochs=epochs, lr=1e-3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plt.sca(axes[0]); plotter(m_histories, metric='loss')
plt.sca(axes[1]); plotter(m_histories, metric='accuracy', ylim=[0.0, 1.0])
plt.tight_layout(); plt.show()


## Step 4 — Read the curves and decide the next move

The from-scratch baseline typically achieves accuracy far below our 90% target on this dataset. Look at the curves — is the model overfitting (train ≫ val), underfitting (both low), or is capacity the bottleneck?

With ~5.9k training images, from-scratch training of a 2M+ parameter network is *data-limited*, not architecture-limited. That points at one of two directions:

1. Get more data (expensive).
2. Use **transfer learning** to get "more data" implicitly, via features learned on ImageNet.

We take direction (2) in **Approach 2**.

## Hyperparameter tuning

Skipped here — for the baseline model, tuning is unlikely to close the gap to 90%. Best to change strategy first (see Approach 2), then tune.

# Approach 2 — Transfer learning

What if we don't have a large dataset to train the model?

We can look for a dataset that has similar characteristics to our task, then either train a model on that task or use a pre-trained model on that task and adapt it to ours.

This is called **transfer learning**. The steps are:

- Select a related task (proxy task) with a large dataset — e.g. ImageNet.
- Train a base model on the proxy task or download a pre-trained one — e.g. VGG / ResNet / MobileNet.
- Identify the feature layer you want to use for your task. Usually the **bottleneck layer** (the one before the final classification layer).
- Remove the layers above the bottleneck and reconfigure to your task. The new top part is called the **head**. Remaining parts from the pre-trained model are the **base model**.
- Freeze all the layers below (and including) the bottleneck layer (weights in the base model).
- Train the network on the new task (cats vs. dogs).
- (Optional) **Fine-tuning**: unfreeze *some* top layers in the frozen base model and train the network for some more epochs on the new task. This will train both the head and part of the base model.

## Establish a new baseline — MobileNetV2 pretrained on ImageNet, base frozen

The classifier head from ImageNet outputs 1000 logits; we replace it with a fresh 2-class linear layer. We freeze `model.features` so only the tiny head (1280 × 2 = 2562 trainable parameters) is updated.

In [ ]:
def build_tfl_model(freeze_base=True, dropout=0.0, num_classes=2):
    m = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2)
    if freeze_base:
        for p in m.features.parameters():
            p.requires_grad = False
    m.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(1280, num_classes),
    )
    return m

model_tfl = build_tfl_model(freeze_base=True, dropout=0.0)

trainable = sum(p.numel() for p in model_tfl.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_tfl.parameters())
print(f'Trainable / total parameters: {trainable:,} / {total:,}  '
      f'(head-only)')


## Train the head

Same optimiser, same 10 epochs, same data. Only the head weights are updated.

In [ ]:
m_histories['Model_tfl'] = train_and_track(model_tfl, train_loader, val_loader, epochs=10, lr=1e-3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plt.sca(axes[0]); plotter(m_histories, metric='loss')
plt.sca(axes[1]); plotter(m_histories, metric='accuracy', ylim=[0.0, 1.0])
plt.tight_layout(); plt.show()


## Hyperparameter tuning

**Informed decisions are crucial, not random ones.**

The information needed for these decisions often appears in learning curves or metrics.

For example, the learning curves for the transfer learning model above indicates **overfitting** (the training loss decreases while the validation loss plateaus or increases — the generalisation gap grows). Overfitting can be mitigated through regularisation techniques such as:

- Dropout
- Weight regularisation (weight decay)
- Data augmentation (already applied, but could be increased)

We'll choose **Dropout** here and tune the dropout rate — a single knob, single change, which lets us read the effect cleanly off the next learning curve.

In [ ]:
DROP_RATE = 0.2
model_tfl = build_tfl_model(freeze_base=True, dropout=DROP_RATE)
trainable = sum(p.numel() for p in model_tfl.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}   (dropout={DROP_RATE})')


Dropout has a hyperparameter — the dropout rate. **That needs to be tuned.** In a lab we'd sweep it (e.g. 0.0, 0.1, 0.2, 0.3, 0.5) and compare validation curves.

In [ ]:
m_histories['Model_tfl_drop'] = train_and_track(model_tfl, train_loader, val_loader, epochs=10, lr=1e-3)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plt.sca(axes[0]); plotter(m_histories, metric='loss')
plt.sca(axes[1]); plotter(m_histories, metric='accuracy', ylim=[0.0, 1.0])
plt.tight_layout(); plt.show()


## (Optional) Fine-tuning the pre-trained weights

In the feature-extraction experiment above, only the head was trained; the ImageNet weights in `model_tfl.features` were frozen.

Fine-tuning unfreezes the *top few* layers of the base model and trains them alongside the head, so the pre-trained features can adapt to our specific task. Two rules of thumb:

1. **Only fine-tune after the head has been trained** with the base model frozen. Otherwise the gradients from a random head will wreck the pretrained weights.
2. **Only unfreeze the top layers.** Lower layers learn generic edge/texture features that generalise; higher layers are more task-specific and benefit most from adaptation.
3. **Use a much lower learning rate.** We're perturbing carefully-learned weights, not learning from scratch.

In [ ]:
# MobileNetV2.features is a Sequential of 19 blocks (indices 0..18).
# The Keras version fine-tunes from layer index 100 out of 156 total layers -
# roughly the last ~5 out of 19 inverted-residual blocks in the PyTorch model.
FINE_TUNE_FROM = 14

for i, block in enumerate(model_tfl.features):
    for p in block.parameters():
        p.requires_grad = (i >= FINE_TUNE_FROM)

trainable = sum(p.numel() for p in model_tfl.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_tfl.parameters())
print(f'Unfroze features[{FINE_TUNE_FROM}:]  ->  trainable / total = {trainable:,} / {total:,}')


Train with a lower learning rate. As you are training a much larger set of parameters and want to *readapt* pre-trained weights rather than overwrite them, it is important to use a lower learning rate.

In [ ]:
base_learning_rate = 1e-4    # 10x smaller than head-only training
m_histories['with_TL_finetune'] = train_and_track(
    model_tfl, train_loader, val_loader, epochs=10, lr=base_learning_rate,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plt.sca(axes[0]); plotter(m_histories, metric='loss')
plt.sca(axes[1]); plotter(m_histories, metric='accuracy', ylim=[0.0, 1.0])
plt.tight_layout(); plt.show()


## Test

Only *now* do we touch the test set. Everything above — every architectural choice, every hyperparameter, every "let's try dropout" — was decided on training and validation curves. The test set is the final, one-shot check that our development-set decisions generalised.

In [ ]:
test_metrics = run_epoch(model_tfl, test_loader, nn.CrossEntropyLoss(), optimizer=None)
for k, v in test_metrics.items():
    print(f'test_{k:9s}: {v:.4f}')


# Analysis

Just looking at the validation or test metrics is not sufficient to understand the model's performance on unseen data. You need to carefully analyse the model's outputs / predictions.

This can be done via tools such as:

- **Confusion matrix analysis** — error proportion per class.
- **Error analysis** — visually inspect misclassified examples to find patterns.
- **Bias detection** — check for systematic errors on subgroups.
- **Feature-space visualisation** — e.g. t-SNE on penultimate features to see whether classes are separable in the model's learned representation.

In [ ]:
# Collect predictions on the test set once, reuse for all analyses below.
model_tfl.eval()
all_true, all_pred, all_prob = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model_tfl(xb)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
        all_true.extend(yb.numpy())
        all_pred.extend(preds)
        all_prob.extend(probs)
all_true = np.array(all_true); all_pred = np.array(all_pred); all_prob = np.array(all_prob)

cm = confusion_matrix(all_true, all_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_label)
disp.plot(cmap=plt.cm.Blues)
plt.show()


In [ ]:
# t-SNE on the frozen-base features of the test set
from sklearn.manifold import TSNE

def extract_features(model, loader):
    model.eval()
    feats, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            f = model.features(xb)
            f = F.adaptive_avg_pool2d(f, 1).flatten(1)
            feats.append(f.cpu().numpy())
            ys.append(yb.numpy())
    return np.concatenate(feats), np.concatenate(ys)

features, labels = extract_features(model_tfl, test_loader)
tsne = TSNE(n_components=2, random_state=42, init='pca', perplexity=30)
features_2d = tsne.fit_transform(features)

plt.figure(figsize=(10, 8))
for i in range(2):
    mask = labels == i
    plt.scatter(features_2d[mask, 0], features_2d[mask, 1], label=class_label[i], alpha=0.6)
plt.legend()
plt.title('t-SNE Visualisation of MobileNetV2 features on the test set')
plt.show()


In [ ]:
# View a few incorrect predictions
incorrect_indices = np.where(all_true != all_pred)[0]

def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

plt.figure(figsize=(10, 10))
for i, idx in enumerate(incorrect_indices[:9]):
    image, label = test_ds[idx]
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(denormalize(image))
    plt.title(f'True: {class_label[all_true[idx]]}  |  Pred: {class_label[all_pred[idx]]}')
    plt.axis('off')
plt.show()


Are there any reasons to be concerned?

- Why does the model make lots of mistakes when animals are in unusual poses / crops / lighting?
- Are the failures concentrated in one class (check the confusion matrix)?
- Do failure cases share any obvious visual pattern? If so, that's your next data-collection or augmentation target — bringing us back to step 4 of the loop.

**Now you have lots of information to make the ultimate judgment.**